# Huggingface pipeline

Install all required packages for the fine-tuning experiment.

In [ ]:
import importlib.metadata
import subprocess
import sys

def install_dep(modules: list):
    for pkg in modules:
        base = pkg.split("[")[0]  # strip extras like [torch]
        try:
            importlib.metadata.distribution(base)
            print(f"{pkg} already installed")
        except importlib.metadata.PackageNotFoundError:
            print(f"Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

modules = ["huggingface_hub",
           "datasets", 
           "transformers", 
           "torch", 
           "numpy", 
           "nvidia-ml-py", 
           "transformers[torch]", 
           "evaluate",
           "ipywidgets",
           "scikit-learn",
           "scipy",
           "accelerate",
           "wandb"]
install_dep(modules)

Login with notebook to Huggingface

In [3]:
from huggingface_hub import notebook_login
notebook_login()

Login to Weights & Biases to track the training run.

In [ ]:
import os
import wandb

os.environ["WANDB_PROJECT"] = "test01"
wandb.login()

Load the MRPC dataset from GLUE — 3 668 training and 408 validation sentence pairs labeled as paraphrase or not.

In [4]:
from datasets import load_dataset

raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

Load the BERT tokenizer, define a tokenisation function for sentence pairs, and create a `DataCollatorWithPadding` for dynamic batching.

In [5]:
from transformers import AutoTokenizer, DataCollatorWithPadding
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenizer_function(item):
    return tokenizer(item["sentence1"],item["sentence2"],truncation=True,padding=True,return_tensors="pt")
data_collactor = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Apply tokenisation to train and validation sets; drop raw text columns, rename `label` → `labels`, and format as PyTorch tensors.

In [6]:
train_dataset = raw_datasets["train"]
train_dataset = train_dataset.map(tokenizer_function,batched=True)
train_dataset = train_dataset.remove_columns(["sentence1","sentence2","idx"])
train_dataset = train_dataset.rename_column("label","labels")
train_dataset.set_format("torch")

validation_dataset = raw_datasets["validation"]
validation_dataset = validation_dataset.map(tokenizer_function,batched=True)
validation_dataset = validation_dataset.remove_columns(["sentence1","sentence2","idx"])
validation_dataset = validation_dataset.rename_column("label","labels")
validation_dataset.set_format("torch")


Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Load pretrained `bert-base-uncased` with a fresh 2-class classification head (the MLM/NSP head weights are discarded).

In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Define training configuration — all defaults; checkpoints go to `test-trainer/`.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    "test-trainer",
    run_name="bert-mrpc-baseline",
    report_to="wandb",
    num_train_epochs=3,
    learning_rate=5e-5,
    logging_strategy="steps",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
)

Assemble the `Trainer` with the model, arguments, datasets, collator, and tokeniser.

Define `compute_metrics` so accuracy/F1 are logged at every evaluation during training, not just at the end.

In [ ]:
import evaluate
import numpy as np

glue_metric = evaluate.load("glue", "mrpc")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return glue_metric.compute(predictions=preds, references=labels)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collactor,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

Run the training loop.

In [10]:
trainer.train()

Step,Training Loss
500,0.524082
1000,0.280596


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1377, training_loss=0.33330006762181147, metrics={'train_runtime': 220.9504, 'train_samples_per_second': 49.803, 'train_steps_per_second': 6.232, 'total_flos': 580916321352960.0, 'train_loss': 0.33330006762181147, 'epoch': 3.0})

Run inference on the validation set and inspect the output shape.

In [11]:
predictions = trainer.predict(validation_dataset)
print(predictions.predictions.shape, predictions.label_ids.shape)

(408, 2) (408,)


Convert logit outputs to predicted class labels, then compute GLUE MRPC accuracy and F1.

In [12]:
import evaluate
import numpy as np

preds = np.argmax(predictions.predictions, axis=-1)

metric = evaluate.load("glue", "mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

{'accuracy': 0.8382352941176471, 'f1': 0.888135593220339}

Close out the W&B run so the summary and curves are finalized.

In [ ]:
wandb.finish()